# Pre-model Oneway Analysis (Template)

Generate oneway plots to analyze feature distributions before modeling.

**Usage**: 
1. Copy this notebook to your working directory
2. Modify the parameters in the "Configuration" cell below
3. Run all cells

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
import os

# Work in current directory - compatible with pip-installed package
work_dir = Path.cwd()

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

# Ensure matplotlib uses non-interactive backend for saving plots
matplotlib.use('Agg')

In [ ]:
import pandas as pd
from pathlib import Path

from ins_pricing.modelling.plotting.diagnostics import plot_oneway
from ins_pricing.cli.utils.cli_common import split_train_test

In [ ]:
# ============================================================
# CONFIGURATION - Modify these parameters for your data
# ============================================================

# Data file (relative to where you run this notebook)
data_path = work_dir / 'Data/od_bc.csv'

# Model name (used for output folder naming)
model_name = 'od_bc'

# Target and weight columns
target_col = 'response'
weight_col = 'weights'

# Feature list - MODIFY THIS FOR YOUR DATA
feature_list = [
    'age_owner', 'gender_owner',
    'plt_zone', 'cheling_year', 'carbrand', 'carkind',
    'cartype', 'grpcode', 'pl', 'power_type',
    'price', 'qualitymax', 'seat_num', 'trans_type',
    'absflag', 'airbagcount', 'btype',
    'lastnum_2y_bi', 'lastnum_2y_ctp', 'cover',
    'L', 'N', 'cover_chg', 'delay_ins',
    'medication_mrk', 'owner_insured', 'tplimit',
    'channel', 'dpt',
    'chd_mileage_day', 'chd_mileage_total',
    'chd_num_atten', 'chd_num_prov',
    'chd_r_unfam_road', 'chd_speed_total', 'chd_time_total', 'chd_triprate_prov'
]

# Categorical features - MODIFY THIS FOR YOUR DATA
# Leave empty [] to auto-detect based on unique values
categorical_features = [
    'gender_owner', 'plt_zone', 'cheling_year', 'carbrand',
    'carkind', 'cartype', 'grpcode', 'pl', 'power_type',
    'price', 'seat_num', 'trans_type', 'absflag', 'airbagcount',
    'btype', 'lastnum_2y_bi', 'lastnum_2y_ctp', 'cover',
    'cover_chg', 'delay_ins', 'medication_mrk', 'owner_insured',
    'channel', 'dpt'
]

# Plot settings
n_bins = 10  # Number of bins for continuous features

# Output directory
output_dir = work_dir / 'Results' / 'plot' / model_name / 'oneway' / 'pre'
output_dir.mkdir(parents=True, exist_ok=True)

# Data split settings (optional - set to None to use all data)
holdout_ratio = 0.25  # Test set ratio, or None to skip splitting
rand_seed = 13

# ============================================================
# LOAD AND PROCESS DATA (No changes needed below)
# ============================================================

print(f"Loading data from: {data_path}")
raw = pd.read_csv(data_path, low_memory=False)

# Handle duplicate columns
if raw.columns.duplicated().any():
    dup_cols = raw.columns[raw.columns.duplicated()].tolist()
    print(f"⚠ Warning: Found {len(dup_cols)} duplicate columns, removing...")
    raw = raw.loc[:, ~raw.columns.duplicated(keep='first')].copy()

# Handle duplicate indices
if raw.index.duplicated().any():
    dup_count = raw.index.duplicated().sum()
    print(f"⚠ Warning: Found {dup_count} duplicate indices, resetting...")
    raw = raw.reset_index(drop=True)

# Remove duplicate feature names
original_feature_count = len(feature_list)
feature_list = list(dict.fromkeys(feature_list))
categorical_features = list(dict.fromkeys(categorical_features))

if len(feature_list) < original_feature_count:
    print(
        f"⚠ Warning: Removed {original_feature_count - len(feature_list)} duplicate features")

# Verify all features exist in data
missing_features = [f for f in feature_list if f not in raw.columns]
if missing_features:
    print(
        f"⚠ Warning: {len(missing_features)} features not found, removing from list...")
    feature_list = [f for f in feature_list if f in raw.columns]
    categorical_features = [
        f for f in categorical_features if f in raw.columns]

raw.fillna(0, inplace=True)
print(f"✓ Data loaded: {len(raw):,} rows, {len(raw.columns)} columns")
print(
    f"✓ Using {len(feature_list)} features ({len(categorical_features)} categorical)")

# Optional: Split data (you can skip this if you want to use all data)
if holdout_ratio is not None and holdout_ratio > 0:
    train_df, test_df = split_train_test(
        raw,
        holdout_ratio=holdout_ratio,
        strategy='random',
        rand_seed=rand_seed,
        reset_index_mode='none',
        ratio_label='holdout_ratio',
    )
    # Use train set for oneway analysis
    df = train_df.reset_index(drop=True).copy()
    print(
        f"✓ Using train split: {len(df):,} rows ({(1-holdout_ratio)*100:.0f}% of data)")
else:
    df = raw.copy()
    print(f"✓ Using all data: {len(df):,} rows")

# Generate oneway plots
print(f"\n{'='*60}")
print(f"Generating oneway plots for {len(feature_list)} features...")
print(f"Output directory: {output_dir}")
print(f"{'='*60}\n")

saved_count = 0
for i, feature in enumerate(feature_list, 1):
    is_categorical = feature in categorical_features

    try:
        save_path = output_dir / f"{feature}.png"

        # Call the plot function
        fig = plot_oneway(
            df,
            feature=feature,
            weight_col=weight_col,
            target_col=target_col,
            n_bins=n_bins,
            is_categorical=is_categorical,
            save_path=str(save_path),
            show=False
        )

        # Verify file was created
        if save_path.exists():
            saved_count += 1
        else:
            print(f"  ⚠ Warning: File not created for {feature}")

        # Progress indicator
        if i % 5 == 0 or i == len(feature_list):
            print(f"  [{i}/{len(feature_list)}] Completed {feature}")

    except Exception as e:
        print(f"  ⚠ Error plotting {feature}: {e}")

print(f"\n{'='*60}")
print(f"✓ Complete! Generated {saved_count}/{len(feature_list)} plots")
print(f"✓ Saved to: {output_dir}")

# Verify output directory contents
if output_dir.exists():
    files = list(output_dir.glob("*.png"))
    print(f"✓ Found {len(files)} PNG files in output directory")
else:
    print(f"⚠ Warning: Output directory does not exist!")

print(f"{'='*60}")